# Training YOLOv8 Hookv3 — 300 Epoch

Dataset: `Hookv3.v1i.yolov8.zip` (Roboflow, kelas `0: Hookv3`).
Berisi 4.748 gambar: train 4.140, valid 392, test 216; label bounding box YOLOv8.
Pilih runtime GPU A100 di Colab, lalu jalankan sel berurutan dan upload ZIP saat diminta.


In [ ]:
%pip -q install ultralytics

import os
import shutil
import zipfile
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

assert torch.cuda.is_available(), 'GPU belum aktif. Pilih Runtime > Change runtime type > A100 GPU.'
DEVICE = 0
GPU_NAME = torch.cuda.get_device_name(DEVICE)
VRAM_GB = torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3
WORKERS = min(8, os.cpu_count() or 1)
print('Torch:', torch.__version__)
print(f'GPU: {GPU_NAME} | VRAM: {VRAM_GB:.1f} GiB | workers: {WORKERS}')
if 'A100' not in GPU_NAME:
    print('GPU yang terpasang bukan A100. Batch otomatis tetap mengikuti kapasitas GPU ini.')

In [ ]:
from google.colab import files

DATASET_ZIP = 'Hookv3.v1i.yolov8.zip'
ZIP_PATH = Path('/content') / DATASET_ZIP
if not ZIP_PATH.is_file():
    print(f'Upload {DATASET_ZIP}')
    uploaded = files.upload()
    assert DATASET_ZIP in uploaded, f'File yang diperlukan: {DATASET_ZIP}'
    ZIP_PATH.write_bytes(uploaded[DATASET_ZIP])
    del uploaded
RAW_ROOT = Path('/content/hookv3_dataset_raw')
if RAW_ROOT.exists():
    shutil.rmtree(RAW_ROOT)
RAW_ROOT.mkdir(parents=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(RAW_ROOT)
print('Dataset diekstrak ke:', RAW_ROOT)
print('Ukuran ZIP (MB):', round(ZIP_PATH.stat().st_size / 1024**2, 1))

In [ ]:
# Pertahankan nama kelas/metadata ekspor; perbaiki path relatif Roboflow untuk Colab.
source_yamls = list(RAW_ROOT.rglob('data.yaml'))
assert len(source_yamls) == 1, f'Harus ada satu data.yaml, ditemukan: {source_yamls}'
DATASET_ROOT = source_yamls[0].parent
config = yaml.safe_load(source_yamls[0].read_text())
assert config['nc'] == 1 and config['names'] == ['Hookv3'], 'Dataset harus berisi kelas 0: Hookv3.'
for split in ('train', 'valid', 'test'):
    assert (DATASET_ROOT / split / 'images').is_dir(), f'Folder {split}/images tidak ditemukan.'
    assert (DATASET_ROOT / split / 'labels').is_dir(), f'Folder {split}/labels tidak ditemukan.'
config.update(path=str(DATASET_ROOT), train='train/images', val='valid/images', test='test/images')
DATA_YAML = Path('/content/hookv3_data.yaml')
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
print(DATA_YAML.read_text())

In [ ]:
# Hookv3 sudah memakai bbox: class_id x_center y_center width height (normalisasi 0–1).
import math

counts = {}
bad = []
for split in ('train', 'valid', 'test'):
    split_dir = DATASET_ROOT / split
    images = [p for p in (split_dir / 'images').iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    labels = list((split_dir / 'labels').glob('*.txt'))
    assert images, f'Tidak ada gambar di split {split}.'
    assert {p.stem for p in images} == {p.stem for p in labels}, f'Pasangan gambar/label tidak cocok di {split}.'
    rows = empty = 0
    for label_path in labels:
        lines = label_path.read_text().strip().splitlines()
        if not lines:
            empty += 1
        for line_no, line in enumerate(lines, 1):
            try:
                values = [float(v) for v in line.split()]
                valid = (
                    len(values) == 5
                    and all(math.isfinite(v) for v in values)
                    and values[0] == 0
                    and all(0 <= v <= 1 for v in values[1:])
                    and values[3] > 0 and values[4] > 0
                )
            except ValueError:
                valid = False
            if not valid:
                bad.append((str(label_path), line_no, line))
            rows += 1
    counts[split] = {'images': len(images), 'label_files': len(labels), 'rows': rows, 'empty': empty}
assert not bad, f'Label bbox invalid: {bad[:3]}'
print(counts)

## Training 300 epoch di Colab A100

Target 300 epoch penuh; `patience=0` menonaktifkan early stopping.
`batch=-1` memilih batch otomatis dengan target sekitar 60% VRAM; `amp=True` mengaktifkan mixed precision.
`cache='ram'` mengurangi pembacaan disk dengan menyimpan gambar di RAM sistem (bukan VRAM).
Jika RAM runtime tidak cukup, gunakan `cache='disk'` atau `cache=False`.
Worker mengikuti jumlah CPU runtime, maksimal 8. Evaluasi test memakai batch tetap 32.
Model `yolov8n.pt`, ukuran gambar 640; hasil di `/content/hookv3_yolo_runs/yolov8n_hookv3_300`.
Gunakan `best.pt`, yaitu checkpoint terbaik menurut validasi selama training.
Durasi dan peningkatan akurasi belum dapat dipastikan sebelum training dijalankan.

Referensi: [pengaturan training Ultralytics](https://docs.ultralytics.com/modes/train/).


In [ ]:
model = YOLO('yolov8n.pt')
results = model.train(
    data=str(DATA_YAML),
    epochs=300, # Jumlah epoch untuk pelatihan
    imgsz=640, # Ukuran gambar input
    batch=-1, # AutoBatch sesuai VRAM GPU (target sekitar 60%)
    device=DEVICE, # GPU Colab
    workers=WORKERS, # Sesuai CPU runtime, maksimal 8
    amp=True, # Mixed precision
    cache='ram', # Kurangi pembacaan gambar dari disk selama training
    patience=0, # Nonaktifkan early stopping: target 300 epoch penuh
    project='/content/hookv3_yolo_runs',
    name='yolov8n_hookv3_300',
    exist_ok=True,
    pretrained=True,
    plots=True,
)
BEST_PT = Path('/content/hookv3_yolo_runs/yolov8n_hookv3_300/weights/best.pt')
assert BEST_PT.exists(), f'Checkpoint tidak ditemukan: {BEST_PT}'
print('Best model:', BEST_PT)

In [ ]:
# Evaluasi pada test split bawaan Hookv3. Generalisasi perlu diuji pada rekaman baru yang independen.
best_model = YOLO(str(BEST_PT))
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=640,
    batch=32,
    device=DEVICE,
    workers=WORKERS,
    project='/content/hookv3_yolo_runs',
    name='yolov8n_hookv3_300_test',
    exist_ok=True,
    plots=True,
)
print(test_metrics.results_dict)

In [ ]:
# Contoh inferensi pada beberapa frame underwater dari test split.
test_sources = sorted((DATASET_ROOT / 'test/images').glob('*.jpg'))[:12]
predictions = best_model.predict(
    source=[str(p) for p in test_sources],
    imgsz=640,
    conf=0.25,
    device=DEVICE,
    save=True,
    project='/content/hookv3_yolo_runs',
    name='yolov8n_hookv3_300_preview',
    exist_ok=True,
)
print('Preview:', '/content/hookv3_yolo_runs/yolov8n_hookv3_300_preview')

In [ ]:
# Download paket laporan lengkap, termasuk best.pt, grafik, CSV, dan preview.
import csv
from google.colab import files

RUN_DIR = BEST_PT.parent.parent
PROJECT_DIR = RUN_DIR.parent
REPORT_DIRS = [RUN_DIR, PROJECT_DIR / f'{RUN_DIR.name}_test', PROJECT_DIR / f'{RUN_DIR.name}_preview']
assert BEST_PT.is_file(), 'Jalankan training terlebih dahulu.'
assert all(p.is_dir() for p in REPORT_DIRS), 'Jalankan training, evaluasi test, dan preview terlebih dahulu.'
assert (RUN_DIR / 'results.csv').is_file(), 'results.csv training tidak ditemukan.'

# Simpan metrik test yang sebelumnya hanya dicetak ke layar.
with (REPORT_DIRS[1] / 'test_metrics.csv').open('w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['metric', 'value'])
    writer.writerows(test_metrics.results_dict.items())

report = [
    '# Laporan Training dan Evaluasi Hookv3',
    '',
    f'- Dataset: {DATASET_ZIP}',
    f'- GPU training: {GPU_NAME} ({VRAM_GB:.1f} GiB)',
    '- Model: YOLOv8n; input 640; target 300 epoch.',
    '- Target lingkungan: kolam kedalaman 1 m, ukuran 10×10 m atau 5×5 m.',
    '',
    '## Jumlah data',
    *[f'- {split}: {info["images"]} gambar, {info["rows"]} bounding box.' for split, info in counts.items()],
    '',
    '## Metrik test',
    '| Metrik | Nilai |',
    '| --- | --- |',
    *[f'| {key} | {float(value):.6f} |' for key, value in test_metrics.results_dict.items()],
    '',
    '## Isi paket',
    f'- {RUN_DIR.name}/: results.png, results.csv, konfigurasi training, grafik yang dihasilkan Ultralytics, dan weights/best.pt serta last.pt.',
    f'- {RUN_DIR.name}_test/: test_metrics.csv dan grafik evaluasi test yang dihasilkan Ultralytics.',
    f'- {RUN_DIR.name}_preview/: contoh gambar prediksi.',
    '- data.yaml: konfigurasi dataset Colab. Path perlu disesuaikan jika dipakai di komputer lain.',
    '',
    'Nama/lokasi grafik mengikuti versi Ultralytics. Lihat juga subfolder hasil.',
    'Metrik ini berasal dari dataset, belum mengukur latensi perangkat ROV, keberhasilan gerakan, atau performa pada kolam target.',
]
REPORT_ZIP = PROJECT_DIR / f'{RUN_DIR.name}_laporan.zip'
with zipfile.ZipFile(REPORT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('LAPORAN.md', '\n'.join(report) + '\n')
    zf.write(DATA_YAML, 'data.yaml')
    for folder in REPORT_DIRS:
        for path in sorted(folder.rglob('*')):
            if path.is_file():
                zf.write(path, path.relative_to(PROJECT_DIR))
print('Paket laporan:', REPORT_ZIP)
print('Ukuran (MB):', round(REPORT_ZIP.stat().st_size / 1024**2, 1))
files.download(str(REPORT_ZIP))


## Target penggunaan: ROV di kolam

Lingkungan yang dituju: kolam kedalaman 1 meter dengan luas 10×10 meter atau 5×5 meter.
Ukuran kolam adalah konteks pengambilan data; bukan parameter kedalaman atau koordinat untuk model deteksi ini.

- Evaluasi dengan rekaman baru dari kamera ROV di dalam air, mencakup kedua ukuran kolam jika keduanya akan digunakan.
- Sertakan hook dekat/jauh dan dari berbagai sudut, pantulan permukaan, bayangan, air keruh, blur gerakan, serta gambar tanpa hook.
- Pisahkan train/valid/test berdasarkan sesi rekaman, agar frame berdekatan dari video yang sama tidak tersebar ke split berbeda.
- Ukur precision, recall, dan kesalahan deteksi menurut jarak/kondisi; tentukan threshold confidence dari validasi tersebut. `conf=0.25` di preview hanya nilai awal.
- Ukur latensi dan FPS pada komputer yang menjalankan deteksi ROV. Kecepatan training A100 tidak mengukur kecepatan inferensi perangkat ROV.

Training 300 epoch dan evaluasi test bawaan belum membuktikan keberhasilan deteksi di kolam target.
